# Part A - Masking  (Notebook 1 of 2)

Locally redacts **confidential branding** from engineering drawings *before* any
data leaves the machine for the cloud extractor.

- Detects the **company name** (text runs -> red boxes in debug) and the **logo**
  (blue box) using a local Qwen3-VL model + Tesseract OCR + PyMuPDF text layer.
- **Blackens** those regions out.
- Hands off **only the masked images** to Part B.

**Output of this notebook**
- `masked_drawings/` - the clean, blackened images (no debug overlays).
- `masked_drawings.zip` - the same set, zipped for transfer to **`PartB_Extraction.ipynb`**.

**Runtime:** needs a **GPU** (Colab T4 is enough; the VLM loads in 4-bit). No API keys required here.

### A0 - Install masking dependencies

In [ ]:
# Masking stack only (local VLM + OCR + PDF). No API SDKs needed in Part A.
!apt-get -qq install -y tesseract-ocr tesseract-ocr-eng poppler-utils
!pip install -q -U transformers accelerate bitsandbytes
!pip install -q PyMuPDF rapidfuzz pytesseract opencv-python-headless
print("Masking dependencies installed.")

## 1 · Global config
Set your input zip name here.

In [ ]:
from pathlib import Path

def _on_colab():
    try:
        import google.colab  # noqa
        return True
    except Exception:
        return False

def _repo_root():
    for d in [Path.cwd(), *Path.cwd().resolve().parents]:
        if (d / "src").is_dir() and (d / "notebooks").is_dir():
            return d
    return Path.cwd()

def _handoff_dir():
    # masking OUTPUT == extraction INPUT (identical on both notebooks)
    d = Path("/content/masked_drawings") if _on_colab() \
        else _repo_root() / "data" / "interim" / "masked_drawings"
    d.mkdir(parents=True, exist_ok=True)
    return d

def _find_raw(name="datasetSampleVLM.zip"):
    for p in [Path.cwd() / name, _repo_root() / "data" / "raw" / name,
              Path("/content") / name, Path.cwd() / "data" / "raw" / name]:
        if p.is_file():
            return str(p)
    return name   # bare name -> upload to /content on Colab

RAW_INPUT_ZIP    = _find_raw()
RAW_DRAWING_DIR  = "drawings" if _on_colab() else str(_repo_root() / "data" / "interim" / "drawings")
MASKED_INPUT_DIR = str(_handoff_dir())
Path(RAW_DRAWING_DIR).mkdir(parents=True, exist_ok=True)
print("RAW_INPUT_ZIP    =", RAW_INPUT_ZIP)
print("RAW_DRAWING_DIR  =", RAW_DRAWING_DIR)
print("MASKED_INPUT_DIR =", MASKED_INPUT_DIR, " (masking output == extraction input)")

## 2 · Unzip the raw drawings

In [ ]:

!unzip -o {RAW_INPUT_ZIP} -d {RAW_DRAWING_DIR}/

import glob
from pathlib import Path
_ex = {".pdf",".png",".jpg",".jpeg",".tif",".tiff",".bmp"}
_files = [p for p in glob.glob(f"{RAW_DRAWING_DIR}/**/*", recursive=True) if Path(p).suffix.lower() in _ex]
print(f"\nFound {len(_files)} raw drawing(s).")
for p in _files[:20]: print("  ", p)

# PART A — Masking

### A1 · Masking imports, constants & VLM prompt

In [ ]:
import os, re, json, glob
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from pathlib import Path
import cv2, numpy as np
import pytesseract
from pytesseract import Output
from rapidfuzz import fuzz
import fitz  # PyMuPDF
import torch
from IPython.display import Image as IPyImage, display

print("GPU:", torch.cuda.is_available(),
      "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"))

IMAGE_EXTS = {".png",".jpg",".jpeg",".tif",".tiff",".bmp"}
COMMON_STOP = {"and","co","of","the","for","is","in","to","by","or","a","an"}
SUFFIXES = {"pvt","ltd","limited","llp","inc","corp","corporation","co","company",
            "technologies","technology","industries","enterprises","solutions",
            "systems","group","gmbh","srl","ag","bv","sa","international","global"}

VLM_PROMPT = (
    "You are analyzing ONE engineering drawing of a mechanical part. "
    "Identify the OWNER COMPANY / MANUFACTURER whose name appears in the title "
    "block (usually beside a logo, and often again in a copyright/IP notice). "
    "Report the COMPLETE company name EXACTLY as printed, including every word and "
    "suffix (e.g. 'GE Energy', 'Lectrix Technologies Pvt Ltd'). Never shorten it to "
    "one word. Do NOT return the part name, drawing title, material, standards, or "
    "person names (drawn / checked / approved by). "
    "Also list any abbreviation or short form of that company shown anywhere on the "
    "sheet (e.g. 'LTPL', 'GE'). "
    "Finally give the bounding box of the company LOGO graphic (the emblem/symbol "
    "itself, not its text) as fractions of the image.\n"
    "Respond with ONLY this JSON and nothing else:\n"
    '{"company_name": "<complete name or null>", '
    '"aliases": ["<abbreviation>", "..."], '
    '"logo_bbox": [x1, y1, x2, y2]}\n'
    "logo_bbox values are fractions 0..1 of width (x) and height (y); "
    "use null if there is no logo, and [] for aliases if none."
)

### A2 · Text localizer helpers

In [ ]:
def norm(t):
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]+", " ", t.lower())).strip()

def derive(names):
    toks = [tok for n in names for tok in norm(n).split() if tok]
    anchors = {t for t in toks if len(t) >= 3 and t not in COMMON_STOP}
    if not anchors:                       # very short company, e.g. "GE"
        anchors = {t for t in toks if len(t) >= 2 and t not in COMMON_STOP}
    extend = set(toks) | SUFFIXES
    return set(toks), anchors, extend

def is_anchor(w, anchors, thr):
    nw = norm(w)
    return bool(nw) and any(fuzz.ratio(a, nw) >= thr for a in anchors)

def is_extend(w, extend, anchors, thr):
    nw = norm(w)
    if not nw:
        return False
    if nw in extend:
        return True
    if is_anchor(w, anchors, thr):
        return True
    return any(len(t) >= 4 and fuzz.ratio(t, nw) >= thr for t in extend)

def _union(bs):
    return (min(b[0] for b in bs), min(b[1] for b in bs),
            max(b[2] for b in bs), max(b[3] for b in bs))

def run_boxes(words, anchors, extend, thr):
    """words: [{'t','b','k'}] -> [(box, text)] of contiguous name runs (whole-word)."""
    lines = {}
    for w in words:
        lines.setdefault(w["k"], []).append(w)
    out = []
    for ws in lines.values():
        ws = sorted(ws, key=lambda w: w["b"][0])
        mark = [False] * len(ws)
        for i, w in enumerate(ws):
            if is_anchor(w["t"], anchors, thr):
                mark[i] = True
                j = i - 1
                while j >= 0 and is_extend(ws[j]["t"], extend, anchors, thr):
                    mark[j] = True; j -= 1
                j = i + 1
                while j < len(ws) and is_extend(ws[j]["t"], extend, anchors, thr):
                    mark[j] = True; j += 1
        run = []
        for i in range(len(ws)):
            if mark[i]:
                run.append(ws[i])
            elif run:
                out.append((_union([w["b"] for w in run]), " ".join(w["t"] for w in run)))
                run = []
        if run:
            out.append((_union([w["b"] for w in run]), " ".join(w["t"] for w in run)))
    return out

def pdf_words(page, scale):
    out = []
    for x0, y0, x1, y1, wd, b, l, n in page.get_text("words"):
        out.append({"t": wd, "b": (int(x0*scale), int(y0*scale),
                                   int(x1*scale), int(y1*scale)), "k": ("pdf", b, l)})
    return out

def ocr_words(bgr):
    d = pytesseract.image_to_data(bgr, output_type=Output.DICT)
    out = []
    for i, txt in enumerate(d["text"]):
        if not txt.strip() or int(d["conf"][i]) < 0:
            continue
        x, y, w, h = d["left"][i], d["top"][i], d["width"][i], d["height"][i]
        out.append({"t": txt, "b": (x, y, x+w, y+h),
                    "k": ("ocr", d["block_num"][i], d["par_num"][i], d["line_num"][i])})
    return out

def black_out(img, box, pad, color):
    h, w = img.shape[:2]
    x0, y0, x1, y1 = box
    cv2.rectangle(img, (max(0, x0-pad), max(0, y0-pad)),
                  (min(w, x1+pad), min(h, y1+pad)), color, -1)

print("localizer ready")

### A3 · PDF logo detector

In [ ]:
def logo_from_pdf(page, name_boxes_pt, scale, page_area):
    """Pick the embedded image that sits on the SAME ROW as the company name."""
    if not name_boxes_pt:
        return None
    ny0 = min(b[1] for b in name_boxes_pt)
    ny1 = max(b[3] for b in name_boxes_pt)
    cands = []
    for im in page.get_images(full=True):
        xref = im[0]
        for r in page.get_image_rects(xref):
            area = r.width * r.height
            if area > 0.03 * page_area or area < 0.00015 * page_area:
                continue                      # skip big part renders and tiny dots
            v_overlap = min(r.y1, ny1) - max(r.y0, ny0)
            h_gap = min((r.x0 - b[2]) if r.x0 >= b[2] else
                        ((b[0] - r.x1) if r.x1 <= b[0] else 0) for b in name_boxes_pt)
            cands.append((v_overlap, -h_gap, r))
    same_row = [c for c in cands if c[0] > 0]
    if not same_row:
        return None
    same_row.sort(reverse=True)
    r = same_row[0][2]
    return (int(r.x0*scale), int(r.y0*scale), int(r.x1*scale), int(r.y1*scale))

print("logo detector ready")

### A4 · Load Qwen3-VL (4-bit) — needs the GPU runtime

In [ ]:
from transformers import (Qwen3VLForConditionalGeneration, AutoProcessor,
                          BitsAndBytesConfig)

MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"   # 17.5 GB bf16 -> 4-bit is required on a T4

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # T4 = fp16 (not bf16)
    bnb_4bit_use_double_quant=True,
)
vlm_model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map="auto")
vlm_processor = AutoProcessor.from_pretrained(MODEL_ID)
print("VLM loaded (4-bit):", MODEL_ID)

### A5 · VLM identify

In [ ]:
def parse_vlm_json(text):
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if not m:
        return None, [], None
    try:
        obj = json.loads(m.group(0))
    except json.JSONDecodeError:
        return None, [], None
    name = obj.get("company_name")
    if isinstance(name, str) and name.strip().lower() in ("", "null", "none"):
        name = None
    aliases = obj.get("aliases") or []
    if not isinstance(aliases, list):
        aliases = []
    aliases = [a for a in aliases if isinstance(a, str) and a.strip()]
    box = obj.get("logo_bbox")
    if (isinstance(box, (list, tuple)) and len(box) == 4
            and all(isinstance(v, (int, float)) for v in box)):
        logo = tuple(float(v) for v in box)
    else:
        logo = None
    return name, aliases, logo

def vlm_identify(bgr):
    from PIL import Image
    h, w = bgr.shape[:2]
    s = min(1.0, VLM_MAX_SIDE / max(h, w))          # downscale for memory + speed
    small = cv2.resize(bgr, (int(w*s), int(h*s))) if s < 1.0 else bgr
    pil = Image.fromarray(cv2.cvtColor(small, cv2.COLOR_BGR2RGB))
    messages = [{"role": "user", "content": [
        {"type": "image", "image": pil}, {"type": "text", "text": VLM_PROMPT}]}]
    inputs = vlm_processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt").to(vlm_model.device)
    with torch.no_grad():
        out = vlm_model.generate(**inputs, max_new_tokens=256, do_sample=False)
    reply = vlm_processor.decode(out[0][inputs["input_ids"].shape[-1]:],
                                 skip_special_tokens=True)
    del inputs, out
    torch.cuda.empty_cache()
    return parse_vlm_json(reply)

print("vlm_identify ready")

### A6 · Redact page

In [ ]:
def redact_page(page, bgr, name, aliases, logo_frac_vlm, *, dpi, threshold, pad,
                fill, no_logo, logo_expand, logo_override, debug):
    H, W = bgr.shape[:2]
    scale = dpi / 72.0
    boxes, name_boxes_px = [], []
    if name:
        toks, anchors, extend = derive([name] + list(aliases))
        words = []
        if page is not None:
            words += pdf_words(page, scale)      # exact text-layer words
        words += ocr_words(bgr)                  # backup for outlined/raster text
        for b, t in run_boxes(words, anchors, extend, threshold):
            boxes.append(b); name_boxes_px.append(b)

    logo_px, logo_src = None, "none"
    if not no_logo:
        if logo_override is not None:
            x1, y1, x2, y2 = logo_override
            logo_px, logo_src = (int(x1*W), int(y1*H), int(x2*W), int(y2*H)), "override"
        elif page is not None and name_boxes_px:
            name_pt = [(b[0]/scale, b[1]/scale, b[2]/scale, b[3]/scale) for b in name_boxes_px]
            logo_px = logo_from_pdf(page, name_pt, scale, page.rect.width*page.rect.height)
            if logo_px:
                logo_src = "pdf"
        if logo_px is None and logo_frac_vlm:
            x1, y1, x2, y2 = logo_frac_vlm
            logo_px, logo_src = (int(x1*W), int(y1*H), int(x2*W), int(y2*H)), "vlm"
        if logo_px is not None:
            ex, ey = int(logo_expand*W), int(logo_expand*H)
            logo_px = (logo_px[0]-ex, logo_px[1]-ey, logo_px[2]+ex, logo_px[3]+ey)
            boxes.append(logo_px)

    masked = bgr.copy()
    debug_img = bgr.copy() if debug else None
    for i, b in enumerate(boxes):
        black_out(masked, b, pad, tuple(fill))
        if debug:
            is_logo = (logo_px is not None and i == len(boxes)-1)
            cv2.rectangle(debug_img, (b[0], b[1]), (b[2], b[3]),
                          (255, 0, 0) if is_logo else (0, 0, 255), 3)
    return masked, debug_img, len(name_boxes_px), logo_src

print("redact_page ready")

### A7 · Masking config

In [ ]:
INPUT_DIR     = RAW_DRAWING_DIR   # raw drawings unpacked in the unzip step above
OUTPUT_DIR    = "out"
DPI           = 300            # PDF rasterization DPI for masking + OCR
THRESHOLD     = 82            # whole-word fuzzy strictness (0-100)
PAD           = 4             # px padding around each blacked box
FILL          = (0, 0, 0)     # BGR (black)
VLM_MAX_SIDE  = 1400          # longest side fed to the VLM (lower if OOM, raise if misread)
LOGO_EXPAND   = 0.006         # grow detected logo box by this fraction per side
NO_LOGO       = False
NAME_OVERRIDE = None          # force a name (single-company runs only)
LOGO_OVERRIDE = None          # force logo box, e.g. (0.49, 0.73, 0.56, 0.80)
DEBUG         = True

### A8 · Run masking

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
files = [Path(p) for p in sorted(glob.glob(f"{INPUT_DIR}/**/*", recursive=True))
         if Path(p).suffix.lower() in IMAGE_EXTS | {".pdf"}]
print(f"Processing {len(files)} drawing(s)...\n")

for f in files:
    is_pdf = f.suffix.lower() == ".pdf"
    doc = fitz.open(str(f)) if is_pdf else None
    pages = list(enumerate(doc)) if is_pdf else [(0, None)]
    for pi, page in pages:
        if is_pdf:
            pix = page.get_pixmap(dpi=DPI)
            arr = np.frombuffer(pix.samples, np.uint8).reshape(pix.height, pix.width, pix.n)
            bgr = cv2.cvtColor(arr, cv2.COLOR_RGB2BGR if pix.n == 3 else cv2.COLOR_RGBA2BGR)
        else:
            bgr = cv2.imread(str(f), cv2.IMREAD_COLOR)
            if bgr is None:
                continue
        name, aliases, logo = NAME_OVERRIDE, [], None
        if NAME_OVERRIDE is None or not NO_LOGO:
            v_name, v_aliases, v_logo = vlm_identify(bgr)
            if name is None:
                name = v_name
            aliases = v_aliases
            logo = v_logo
        masked, dbg, n_name, logo_src = redact_page(
            page, bgr, name, aliases, logo,
            dpi=DPI, threshold=THRESHOLD, pad=PAD, fill=FILL, no_logo=NO_LOGO,
            logo_expand=LOGO_EXPAND, logo_override=LOGO_OVERRIDE, debug=DEBUG)
        stem = f"{f.stem}_p{pi+1}"
        cv2.imwrite(f"{OUTPUT_DIR}/{stem}_masked.png", masked)
        if DEBUG and dbg is not None:
            cv2.imwrite(f"{OUTPUT_DIR}/{stem}_debug.png", dbg)
        flag = "  !! NAME NOT FOUND" if not name else ""
        print(f"[{f.name} p{pi+1}] name={name!r} aliases={aliases} "
              f"name_runs={n_name} logo={logo_src} -> {stem}_masked.png{flag}")
    if doc:
        doc.close()
print("\nDone.")

### A9 · Verify — show debug overlays AND blackened (masked) outputs
- **Debug overlays** (red = text boxes, blue = logo box) — visual check only, not passed downstream.
- **Blackened / masked images** — the actual sheets handed off to Part B extraction.

In [ ]:
print("=" * 80)
print("DEBUG OVERLAYS  (red = text boxes, blue = logo box)")
print("=" * 80)
for p in sorted(glob.glob(f"{OUTPUT_DIR}/*_debug.png")):
    print(p)
    display(IPyImage(filename=p, width=1000))

print()
print("=" * 80)
print("BLACKENED / MASKED OUTPUTS  (these are the sheets fed into Part B)")
print("=" * 80)
for p in sorted(glob.glob(f"{OUTPUT_DIR}/*_masked.png")):
    print(p)
    display(IPyImage(filename=p, width=1000))

## Hand-off - keep ONLY the masked images and zip them for Part B

The debug overlays are dropped. The clean masked set is copied into
`masked_drawings/` and zipped to `masked_drawings.zip`. On Colab the zip is also
downloaded so you can upload it into Part B (or just run Part B in the same
runtime, where it will pick up the folder automatically).

In [ ]:
import shutil, glob, os, gc, torch

STRIP_MASKED_SUFFIX = True   # rename "<stem>_masked.png" -> "<stem>.png" for the hand-off

# fresh hand-off folder
os.makedirs(MASKED_INPUT_DIR, exist_ok=True)
for _old in glob.glob(f"{MASKED_INPUT_DIR}/*"):
    os.remove(_old)

masked_files = sorted(glob.glob(f"{OUTPUT_DIR}/*_masked.png"))
debug_files  = glob.glob(f"{OUTPUT_DIR}/*_debug.png")

for _src in masked_files:
    base = os.path.basename(_src)
    if STRIP_MASKED_SUFFIX:
        base = base.replace("_masked.png", ".png")
    shutil.copy(_src, os.path.join(MASKED_INPUT_DIR, base))

print(f"Handed off {len(masked_files)} masked image(s) -> {MASKED_INPUT_DIR}")
print(f"Excluded {len(debug_files)} debug image(s) from the hand-off.")

# zip the clean set for transfer to Part B (Notebook 2)
if os.path.exists("masked_drawings.zip"):
    os.remove("masked_drawings.zip")
shutil.make_archive("masked_drawings", "zip", MASKED_INPUT_DIR)
print("Hand-off archive ready: masked_drawings.zip")

# free the local VLM from GPU memory (Part B is API-only, in the other notebook)
for _v in ("vlm_model", "vlm_processor"):
    if _v in globals():
        del globals()[_v]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("VLM unloaded; GPU memory freed.")

# on Colab, download the zip so it can be uploaded into Part B
try:
    from google.colab import files as colab_files
    colab_files.download("masked_drawings.zip")
except Exception as e:
    print("Download step skipped (not on Colab):", e)